# 第11周 · 作业（预计约7小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 用自己写的模块搭 Transformer 分类器（含 padding 掩码），在 IMDB 上与 TF-IDF、TextCNN 比较 | 3.5 小时 |
| B | 注意力热图：模型在看哪些词 | 1 小时 |
| C | 序列反转：有 / 无位置编码各训练一次并解释差异 | 1.5 小时 |
| D | 代码讲解段落 + 反思 | 1 小时 |

**规则**：可以使用 AI 编程助手，但 ① **禁用** `nn.TransformerEncoder`、`nn.TransformerEncoderLayer` 和 `nn.MultiheadAttention`——注意力、多头、编码器块、位置编码都必须是你自己写的模块（A 部分有一格会自动检查）；② D 部分必须附**代码讲解段落**：用自己的话逐段说明代码在做什么、关键张量的 shape 是什么。面谈时老师会随机抽一段请你现场解释并做一处小修改。

蓝本仍是 UvA Deep Learning Tutorial 6（Phillip Lippe，[MIT](https://github.com/phlippe/uvadlc_notebooks)）。每行张量运算旁边写 shape 注释。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from zero2researcher import data, text, experiments
from zero2researcher.checks import week11 as ck
from zero2researcher.checks import report

# 课程统一的 device 选择：有 CUDA 用 CUDA（Colab GPU），Apple 芯片用 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## A · 自己的 Transformer 分类器

### A.1 四块积木

把课上的四个模块写出来（可以对照自己的课堂笔记，但每一行都要能讲清楚）。接口约定与课上相同：

- `scaled_dot_product(q, k, v, mask=None)` 返回 `(values, attention)`；mask 中 1 = 保留、0 = 遮住；
- `MultiheadAttention(embed_dim, num_heads)`：`qkv_proj` 合并投影，`forward(x, mask=None, return_attention=False)`；
- `EncoderBlock(input_dim, num_heads, dim_feedforward, dropout)`：Post-LN，`forward(x, mask=None)`；
- `PositionalEncoding(d_model, max_len)`：正弦位置编码，`forward(x)` 返回 `x + PE`。

In [ ]:
def scaled_dot_product(q, k, v, mask=None):
    """q, k: (..., T, d_k)；v: (..., T, d_v)；mask 可广播到 (..., T, T)。返回 (values, attention)"""
    # 在这里写你的代码
    raise NotImplementedError


class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x, mask=None, return_attention=False):
        B, T, C = x.shape
        # 在这里写你的代码
        raise NotImplementedError
        return (o, attention) if return_attention else o


class EncoderBlock(nn.Module):
    def __init__(self, input_dim, num_heads, dim_feedforward, dropout=0.0):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x, mask=None):
        # 在这里写你的代码
        raise NotImplementedError


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)       # (1, max_len, C)

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]                                  # (B, T, C)

In [ ]:
ck.scaled_dot_product(scaled_dot_product)
ck.multihead_shapes(MultiheadAttention)

### A.2 带 padding 掩码的分类器

课上的 `TransformerClassifier` 没管 `<pad>`。IMDB 影评按 256 个 token 截断、短的在末尾补 0，平均约四分之一的位置是 pad——不处理的话，真实 token 会把注意力分给 pad，mean-pool 也会把一堆 pad 向量平均进来。你要在两处用上掩码：

1. **注意力**：`keep = (x != 0)` 得到 `(B, T)` 的布尔矩阵，变形为 `(B, 1, 1, T)` 传给每个块；
2. **mean-pool**：只对真实 token 求平均：$\text{pooled} = \dfrac{\sum_t h_t \cdot \text{keep}_t}{\sum_t \text{keep}_t}$。

推荐配置（本周自查区间按它实测）：2 层、2 头、$d_{\text{model}} = 64$、$d_{ff} = 128$、dropout 0.1，embedding 之后也加一层 dropout。`forward` 支持 `return_attention=True`：额外返回**最后一层**的注意力权重 `(B, H, T, T)`，B 部分画热图要用。

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=2, num_layers=2, d_ff=128,
                 num_classes=2, dropout=0.1, max_len=512, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos = PositionalEncoding(d_model, max_len)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout)
                                     for _ in range(num_layers)])
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x, return_attention=False):          # x: (B, T)
        # 在这里写你的代码
        raise NotImplementedError
        return (logits, attn) if return_attention else logits

两道自动检查：① 模型里没有被禁用的 PyTorch 内置模块；② **padding 不变性**——同一条影评后面多补 20 个 pad，输出必须不变（这是检验掩码写对的最直接办法：两处掩码少一处，这条都过不了）。

In [ ]:
torch.manual_seed(0)
probe = TransformerClassifier(vocab_size=100).eval()
banned = (nn.MultiheadAttention, nn.TransformerEncoder, nn.TransformerEncoderLayer)
report("没有使用被禁用的内置模块", not any(isinstance(m, banned) for m in probe.modules()),
       "模型里出现了 nn.MultiheadAttention / nn.TransformerEncoder(Layer)",
       "注意力、多头和编码器块都要用你自己写的类")

short = torch.tensor([[5, 17, 42, 8, 99, 3]])                  # (1, 6)：一条 6 个 token 的"影评"
padded = torch.cat([short, torch.zeros(1, 20, dtype=torch.long)], dim=1)   # (1, 26)：后面补 20 个 pad
with torch.no_grad():
    gap = (probe(short) - probe(padded)).abs().max().item()
report("padding 不变性", gap < 1e-5, f"补 pad 之后 logits 变了（最大差 {gap:.2e}）",
       "检查两处：注意力的 mask 形状是否为 (B, 1, 1, T)；mean-pool 是否只除以真实 token 的个数")

### A.3 训练与评估

数据用 `text.imdb_arrays(max_len=256)`（课程工具包提供的词级分词：词表只在 5000 条训练影评上建，最多 2 万词，id 0 = `<pad>`、1 = `<unk>`）。配置：Adam 1e-3、batch 64、`experiments.set_seed(42)`、**8 个 epoch**。

两条研究纪律：epoch 数事先定死，**不看测试集挑 epoch**；测试集只在训练结束后评估一次。训练循环用第 8 周的 `train_one_epoch` / `evaluate`（再凭记忆写一遍）。

In [ ]:
X_train, y_train, X_test, y_test, vocab = text.imdb_arrays(max_len=256)
X_train, y_train = torch.tensor(X_train), torch.tensor(y_train)
X_test, y_test = torch.tensor(X_test), torch.tensor(y_test)
print("训练", tuple(X_train.shape), "测试", tuple(X_test.shape), "词表", len(vocab))
print(f"pad 占全部位置的比例: {(X_train == 0).float().mean():.2f}")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch，返回平均训练损失"""
    # 在这里写你的代码
    raise NotImplementedError


def evaluate(model, loader, device):
    """返回准确率"""
    # 在这里写你的代码
    raise NotImplementedError


In [ ]:
experiments.set_seed(42)
model = TransformerClassifier(vocab_size=len(vocab), max_len=256).to(device)
print(f"参数量 {sum(p.numel() for p in model.parameters()):,}")
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
t0 = time.perf_counter()
for epoch in range(8):
    losses.append(train_one_epoch(model, train_loader, criterion, optimizer, device))
    print(f"epoch {epoch + 1}: train loss {losses[-1]:.4f}   （累计 {time.perf_counter() - t0:.0f} 秒）")

transformer_acc = evaluate(model, test_loader, device)
print(f"Transformer 测试准确率: {transformer_acc:.4f}")
ck.imdb_transformer_accuracy(transformer_acc)

### A.4 三方对比

同一份 5000 / 2000 划分上的三个模型。TF-IDF + 逻辑回归是第 6 周的基线（这里现场重跑，几秒钟）；TextCNN 的数从第 10 周**课上**（S3 末尾）保存的 `outputs/imdb_results.csv` 里读——3/4/5 多宽度的 TextCNN 是在第 10 周 lecture 里训练的，第 10 周作业只训练了单一宽度的 CNN。在 Colab 上要先把这个 CSV 上传到本 notebook 旁边的 `outputs/` 文件夹；找不到它时，下面先用调研时本机 5 个 epoch 的参考值 0.766，并打印提示。

最后把本周的 Transformer 追加成新的一行、写回同一个文件——第 13–14 周会接着往这张表里加。

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_df, test_df = data.load_imdb("train"), data.load_imdb("test")
vec = TfidfVectorizer(min_df=2)
tfidf_acc = LogisticRegression(max_iter=1000).fit(vec.fit_transform(train_df["text"]), train_df["label"]) \
    .score(vec.transform(test_df["text"]), test_df["label"])

board_path = os.path.join("outputs", "imdb_results.csv")    # 第 10 周课上（S3 末尾）保存的跨周对比表
board = pd.read_csv(board_path) if os.path.exists(board_path) else None
textcnn_rows = None if board is None else board[board["model"].astype(str).str.startswith("TextCNN")]
if textcnn_rows is not None and len(textcnn_rows) > 0:
    textcnn_acc = float(textcnn_rows["test_acc"].iloc[-1])
    print(f"读到 {board_path}：{textcnn_rows['model'].iloc[-1]} = {textcnn_acc:.4f}")
else:
    textcnn_acc = 0.766    # 参考值：调研时本机 5 个 epoch 的 TextCNN（不是你自己的数）
    print(f"没在 {board_path} 里找到 TextCNN 那一行，先用参考值 {textcnn_acc}。"
          "想用自己的数：把第 10 周 lecture 保存的 imdb_results.csv 放进本 notebook 旁边的 outputs/ 文件夹再重跑这一格")
    if board is None:
        board = pd.DataFrame([{"model": "TF-IDF + LR", "test_acc": round(tfidf_acc, 4), "week": 6}])

names = ["TF-IDF + LR", "TextCNN (week 10)", "Transformer (ours)"]
accs = [tfidf_acc, textcnn_acc, transformer_acc]
for n_, a in zip(names, accs):
    print(f"{n_:20s} {a:.4f}")

# 把本周的 Transformer 追加进跨周对比表并写回（重复运行时不会出现两行同名）
week11_row = pd.DataFrame([{"model": "Transformer (from scratch, 8 epochs)",
                            "test_acc": round(transformer_acc, 4), "week": 11}])
board = pd.concat([board, week11_row], ignore_index=True).drop_duplicates("model", keep="last")
os.makedirs("outputs", exist_ok=True)
board.to_csv(board_path, index=False)
print(board.to_string(index=False))
print("已写回", board_path)

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**问题**（4–6 句话）：从零训练的 Transformer 为什么没打过 TF-IDF？请至少从两个角度谈：训练数据量与参数量（135 万参数里多少在 embedding 上？5000 条影评够喂饱吗）；TF-IDF + 逻辑回归对"情感分类"这个任务内置了什么好用的假设（第 10 周"归纳偏置"）。再预测一下：如果模型在**大量无标注文本**上先学过词的含义，结果会怎样？

（提醒：从零训练的小 Transformer 在 5000 条影评上输给 TF-IDF 是**正常现象**，不说明你的代码写错了——第 10 周的 TextCNN 也是如此。第 13–14 周我们会带着预训练模型回到这张表。）

（在这里回答）

## B · 注意力热图：模型在看哪些词

挑一条**短**影评（不超过 50 个 token、且模型分类正确），取**最后一层、第 0 个头**的注意力权重，画成 token × token 的热图（坐标轴标上词）。再算每个词"收到的平均注意力"（热图按列取平均），列出最高的 5 个词。

提示：`itos = {i: w for w, i in vocab.items()}` 把 id 换回词；`model(x, return_attention=True)` 返回 `(logits, attn)`，`attn` 的 shape 是 `(1, H, T, T)`；只画真实 token（去掉末尾的 pad）。

In [ ]:
itos = {i: w for w, i in vocab.items()}
lengths = (X_test != 0).sum(dim=1)                       # (N,)：每条影评的真实 token 数
model.eval()
# 在这里写你的代码
raise NotImplementedError


**问题**（3–4 句话）：模型最关注的词有道理吗（是不是情感词、否定词）？注意力权重能不能直接当成"模型做出判断的理由"？（提示：注意力只是第二层里的一个中间量，后面还有 FFN、残差和另一个头；可搜索 Jain & Wallace 2019, *Attention is not Explanation*。）

（在这里回答）

## C · 序列反转：有 / 无位置编码

用你 A.1 的积木，照课上 S4 的设定各训练一次：5 万条训练序列（`seed=0`）、1 万条测试序列（`seed=1`），每条 16 个 0–9 的数字；模型 = Embedding(10, 32) + [位置编码] + 1 个 `EncoderBlock(32, 1, 64)` + Linear(32 → 10)；Adam 1e-3、batch 128、3 个 epoch、`set_seed(42)`。报告两次的**测试逐位置准确率**。

In [ ]:
def make_reverse_data(n, seq_len=16, num_digits=10, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(num_digits, (n, seq_len), generator=g)    # (N, 16)
    return x, x.flip(1)


Xr_train, Yr_train = make_reverse_data(50000, seed=0)
Xr_test, Yr_test = make_reverse_data(10000, seed=1)


class ReversalModel(nn.Module):
    def __init__(self, use_pe=True, num_digits=10, d_model=32, num_heads=1, d_ff=64):
        super().__init__()
        self.embed = nn.Embedding(num_digits, d_model)
        self.pos = PositionalEncoding(d_model) if use_pe else nn.Identity()
        self.block = EncoderBlock(d_model, num_heads, d_ff)
        self.out = nn.Linear(d_model, num_digits)

    def forward(self, x):                                       # (B, T)
        return self.out(self.block(self.pos(self.embed(x))))    # (B, T, 10)


def train_reversal(use_pe, epochs=3, seed=42):
    """训练 ReversalModel，返回测试集逐位置准确率"""
    experiments.set_seed(seed)
    model = ReversalModel(use_pe=use_pe).to(device)
    # 在这里写你的代码
    raise NotImplementedError


acc_with_pe = train_reversal(use_pe=True)
acc_without_pe = train_reversal(use_pe=False)
print(f"有位置编码 {acc_with_pe:.4f}   无位置编码 {acc_without_pe:.4f}")
ck.reversal_accuracy_with_pe(acc_with_pe)
ck.reversal_accuracy_without_pe(acc_without_pe)

**问题**（4–6 句话）：解释两次结果的差异。要求用上"置换等变"这个词；说明无位置编码时模型**最多**能做到什么、为什么它的准确率不是 0.1（瞎猜）而是 0.24 左右；最后说一说：IMDB 情感分类对位置编码的依赖，会比序列反转强还是弱？为什么？（时间允许，可以把 A 部分的位置编码去掉重训一次验证你的猜想，写进实验记录。）

（在这里回答）

## D · 代码讲解段落 + 反思

**代码讲解**（AI 助手政策要求，必做）：任选你本周写的**一个类**（`MultiheadAttention` / `EncoderBlock` / `TransformerClassifier` 之一），用 6–10 句话逐段讲解：每一段代码在做什么、输入输出张量的 shape、为什么要这样写（例如为什么要 permute、mask 为什么是这个形状）。如果你用了 AI 助手，写明它帮你做了哪一部分、你是怎么验证它写对的。

**反思**（2–3 句话）：本周哪一个 shape 让你卡得最久？你最后是靠什么定位出来的？

（在这里写）